# Tutorial 1: spikes, stimulus, STA and the LN model

This notebook walks through the first half of the analysis for one cell:

1. **Spikes**: what the recorded spike counts look like.
2. **Stimulus**: the flickering checkerboard the retina saw.
3. **Spike-triggered average (STA)**: the average stimulus before a spike, which shows the cell's receptive field in space and time.
4. **Ellipse crop**: cutting the stimulus down to the receptive field.
5. **Linear-nonlinear (LN) model**: predicting the firing rate from the STA, and testing that prediction on a repeated "frozen" stimulus.

Tutorials 2 and 3 build on this to find the receptive field's **subunits** and test whether a subunit model predicts better than the LN model.

Run the cells in order. Everything slow is cached under `results/cache/`, so a second run takes seconds.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

# Run from the repository root whichever folder Jupyter starts in, so the
# relative paths below and `import src` work.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import h5py
import numpy as np
from matplotlib import pyplot as plt

from src.analysis.model_comparison import correlation_ceiling, prediction_metrics
from src.data_cache import cropped_stimulus, load_spike_data
from src.prediction.sta_model_predictions import (
    binned_response,
    gen_sta_model,
    get_sta_predictions,
    ln_drive,
    nonlin,
)
from src.simulation.generate_training_data import crop_sta, gen_sta
from src.visualization.plotting_fxns import (
    MODEL_COLORS,
    OKABE_ITO,
    plot_crop,
    plot_nonlinearity,
    plot_predictions,
    save_figure,
    use_style,
)

use_style()

In [ ]:
# The data folder can live anywhere: set RGC_DATA_ROOT before starting Jupyter
# (e.g. a share or an external drive); otherwise ./data is used.
DATA_ROOT = Path(os.environ.get("RGC_DATA_ROOT", "data"))
DATA_PATH = DATA_ROOT / "cell_data_01_NC.mat"
STIMULUS_PATH = DATA_ROOT / "stimulus_data"
CACHE_DIR = Path("results/cache")          # disposable: delete it to recompute
STA_PATH = CACHE_DIR / "sta"
FIGURES = Path("results/tutorials/01_sta_and_ln_model")

CELL = 0
MAX_TRIALS = None   # None uses all 219 trials; e.g. 20 for a quick first look
WIDTH, HEIGHT, REFRESH_RATE = 200, 150, 30   # stimulus pixels and frames per second
DT = 1 / REFRESH_RATE
FILTER_LENGTH = int(0.67 * REFRESH_RATE)     # 20 frames ≈ 0.67 s of stimulus history
SIGMA = 3                                    # ellipse size, in standard deviations

## 1. Spikes

`cell_data_01_NC.mat` holds two arrays of spike counts, one count per cell per stimulus frame (1/30 s):

- **`spk1`**: responses to *running* white noise, a different random stimulus on every trial. Shape: cells × frames × trials. This is what every model is fitted to.
- **`spk2`**: responses to *frozen* noise, one short stimulus shown again and again. Shape: cells × frames × repeats. Averaging the repeats gives a clean firing rate to test predictions on; the models never see these responses during fitting.

In [ ]:
spikes = load_spike_data(DATA_PATH, CACHE_DIR / "spikes")
spk1, spk2 = spikes["spk1"], spikes["spk2"]
print(f"spk1 (running noise): {spk1.shape}  cells × frames × trials")
print(f"spk2 (frozen noise):  {spk2.shape}  cells × frames × repeats")

num_trials = spk1.shape[2] if MAX_TRIALS is None else min(MAX_TRIALS, spk1.shape[2])
spikecounts = spk1[CELL][:, :num_trials]          # frames × trials for this cell
num_frames = spikecounts.shape[0]
rates = spk1.sum(axis=(1, 2)) / (spk1.shape[1] * spk1.shape[2] * DT)
print(f"Cell {CELL}: {rates[CELL]:.1f} Hz; a spike in {np.mean(spikecounts > 0):.1%} of frames; "
      f"using {num_trials} trials of {num_frames * DT:.0f} s")

Spikes are rare: most frames have none. That will matter in Tutorial 2, where a classifier has to tell spike frames from no-spike frames.

Below, left: every cell's mean firing rate, with the chosen cell highlighted. Right: the chosen cell's responses to the frozen noise. Each row of the raster is one repeat of the same stimulus; spikes line up in columns because the cell responds reliably to the same stimulus. The trace underneath is the repeat-averaged firing rate the models will try to predict.

In [ ]:
fig, (ax_rates, ax_raster, ax_rate) = plt.subplots(
    1, 3, figsize=(12, 3.2), gridspec_kw={"width_ratios": [1.2, 1.5, 1.5]})
order = np.argsort(rates)[::-1]
colors = [OKABE_ITO["vermillion"] if c == CELL else "0.6" for c in order]
ax_rates.bar(np.arange(len(rates)), rates[order], color=colors, width=1.0)
ax_rates.set(xlabel="Cell (Sorted)", ylabel="Firing Rate (Hz)", title="All Cells")

frozen = spk2[CELL]                                # frames × repeats
time_frozen = np.arange(frozen.shape[0]) * DT
shown = min(40, frozen.shape[1])
rows, cols = np.nonzero(frozen[:, :shown].T)
ax_raster.plot(time_frozen[cols], rows, "|", color="k", markersize=3)
ax_raster.set(xlabel="Time (s)", ylabel="Repeat", title=f"Cell {CELL} Frozen Noise")
ax_rate.plot(time_frozen, frozen.mean(axis=1) / DT, color=MODEL_COLORS["Measured"], lw=1)
ax_rate.set(xlabel="Time (s)", ylabel="Firing Rate (Hz)",
            title=f"Mean Of {frozen.shape[1]} Repeats")
fig.tight_layout()
save_figure(fig, FIGURES / "Firing Rates And Frozen Raster", caption=(
    f"Left: mean firing rate of every cell during running noise, sorted; cell {CELL} in "
    f"vermillion. Middle: spikes of cell {CELL} on the first {shown} repeats of the frozen "
    f"noise. Right: firing rate averaged over all {frozen.shape[1]} repeats."))

## 2. Stimulus

Each trial's stimulus is stored in `stimulus_data/stim####.h5` as a pixels × frames array of +1 (white) and −1 (black). Every pixel flips independently at random on every frame: **binary white noise**. Because it contains no structure of its own, any structure in what precedes a spike comes from the cell.

The 30,000 pixel values of each frame unfold into a 200 × 150 image with `reshape(200, 150, frames)`, the layout all the code uses.

In [ ]:
with h5py.File(STIMULUS_PATH / "stim0000.h5", "r") as handle:
    trial0 = handle["stim"][:]
print(f"trial 0: {trial0.shape} (pixels × frames), dtype {trial0.dtype}, values {np.unique(trial0)}")
frames = trial0.reshape(WIDTH, HEIGHT, -1)

fig, axes = plt.subplots(1, 3, figsize=(10, 2.8))
for ax, frame in zip(axes, (0, 1, 2)):
    ax.imshow(frames[:, :, frame].T, origin="lower", cmap="gray", vmin=-1, vmax=1)
    ax.set(title=f"Frame {frame} ({frame * DT * 1000:.0f} ms)", xticks=[], yticks=[])
fig.tight_layout()
save_figure(fig, FIGURES / "Stimulus Frames", caption=(
    "Three consecutive frames of the binary white-noise stimulus (trial 0). "
    "Each of the 200 × 150 pixels is independently white (+1) or black (-1) on each frame."))

## 3. Spike-triggered average (STA)

For every spike, take the 20 frames (0.67 s) of stimulus that came just before it, and average them, weighting by the number of spikes in the frame. Pixels and moments that tend to drive the cell stand out; everything else averages to zero.

The result is a **spatiotemporal** filter: 30,000 pixels × 20 time lags. Two slices of it are easier to look at:

- the **spatial STA**, the frame at the moment of strongest response (the receptive field's shape);
- the **temporal STA**, the time course at the strongest pixel.

`gen_sta` reads every trial's stimulus once, which is the slowest step; the result is cached under `results/cache/sta/`, keyed on exactly the spikes and stimulus files used, so it is only computed once.

In [ ]:
sta, spatial_sta, temporal_sta = gen_sta(
    WIDTH, HEIGHT, spikecounts, CELL, FILTER_LENGTH, num_frames, num_trials,
    sta_path=STA_PATH, stim_path=STIMULUS_PATH,
)
print(f"STA {sta.shape} (pixels × lags), spatial {spatial_sta.shape}, temporal {temporal_sta.shape}")

# Column l of the STA is the frame (FILTER_LENGTH - l) frames before the spike.
lag_times = (np.arange(FILTER_LENGTH) - FILTER_LENGTH) * DT
peak = np.unravel_index(np.abs(spatial_sta).argmax(), spatial_sta.shape)
print(f"Receptive-field peak at pixel {peak}; "
      f"{'ON' if spatial_sta[peak] * temporal_sta[np.abs(temporal_sta).argmax()] > 0 else 'OFF'}-like polarity")

In [ ]:
fig = plt.figure(figsize=(12, 5.5))
grid = fig.add_gridspec(2, 6, height_ratios=[1.4, 1])
ax_space = fig.add_subplot(grid[0, :3])
limit = np.abs(spatial_sta).max()
ax_space.imshow(spatial_sta.T, origin="lower", cmap="bwr", vmin=-limit, vmax=limit)
ax_space.set(title="Spatial STA", xlabel="X (Pixels)", ylabel="Y (Pixels)")
ax_time = fig.add_subplot(grid[0, 3:])
ax_time.axhline(0, color="0.8", lw=0.8)
ax_time.plot(lag_times, temporal_sta, "o-", color=OKABE_ITO["blue"], markersize=3)
ax_time.set(title="Temporal STA", xlabel="Time Before Spike (s)", ylabel="Weight (a.u.)")

cube = sta.reshape(WIDTH, HEIGHT, FILTER_LENGTH)
half = 15
x0, y0 = max(peak[0] - half, 0), max(peak[1] - half, 0)
cube_limit = np.abs(cube).max()
for ax, lag in zip([fig.add_subplot(grid[1, i]) for i in range(6)],
                   np.linspace(FILTER_LENGTH - 1, 4, 6).astype(int)):
    ax.imshow(cube[x0:x0 + 2 * half, y0:y0 + 2 * half, lag].T, origin="lower", cmap="bwr",
              vmin=-cube_limit, vmax=cube_limit)
    ax.set(title=f"{lag_times[lag] * 1000:.0f} ms", xticks=[], yticks=[])
fig.tight_layout()
save_figure(fig, FIGURES / "Spike-Triggered Average", caption=(
    f"Spike-triggered average of cell {CELL} from {num_trials} trials. Top left: spatial STA "
    "at the lag of strongest response (red = white drives the cell, blue = black). Top right: "
    "temporal STA at the peak pixel. Bottom: a 30 × 30 pixel window around the peak at six "
    "lags before the spike."))

## 4. Ellipse crop

Almost all of the 30,000 pixels are noise; only the receptive field matters. A 2-D Gaussian is fitted to the spatial STA, and an ellipse drawn `SIGMA` standard deviations from its centre. The **mask** holds every pixel whose centre lies inside the ellipse; the **crop** is the rectangle around it. From here on, every model sees only the pixels inside the mask.

In [ ]:
cropped_sta, crop, ellipse_mask = crop_sta(spatial_sta, SIGMA)
crop_shape = cropped_sta.shape
print(f"crop {crop} → {crop_shape[0]} × {crop_shape[1]} pixels; {ellipse_mask.sum()} inside the ellipse")

fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(10, 3.8))
plot_crop(spatial_sta, crop, ellipse_mask, ax=ax_full)
ax_full.set_title("Spatial STA With Crop")
plot_crop(spatial_sta, crop, ellipse_mask, ax=ax_zoom)
pad = 8
ax_zoom.set(xlim=(crop[0].start - pad, crop[0].stop + pad),
            ylim=(crop[1].start - pad, crop[1].stop + pad), title="Zoom")
fig.tight_layout()
save_figure(fig, FIGURES / "Ellipse Crop", caption=(
    f"Spatial STA of cell {CELL} with the ellipse mask (green, {SIGMA} standard deviations of "
    "the fitted Gaussian) and the crop rectangle (black). Only pixels inside the ellipse "
    "enter the models."))

Now cut every trial's stimulus down to the crop. `cropped_stimulus` reads each trial file once and keeps just these pixels, trial first; with `CACHE_DIR` set, the result is saved under `results/cache/stimuli/` and re-opened instantly next time (it is about 150 MB for this cell at 219 trials).

In [ ]:
stimuli = cropped_stimulus(
    STIMULUS_PATH, crop, WIDTH, HEIGHT, num_frames, num_trials, CACHE_DIR / "stimuli")
print(f"cropped stimulus: {stimuli.shape} (trials × crop pixels × frames), "
      f"{stimuli.nbytes / 1e6:.0f} MB instead of {num_trials * trial0.nbytes / 1e9:.1f} GB")

## 5. The linear-nonlinear (LN) model

The LN model is the standard description of a ganglion cell:

1. **Linear**: filter the recent stimulus with the cell's own spatiotemporal STA (cropped). The result, the *linear drive*, is one number per frame: how much the stimulus looked like the STA.
2. **Nonlinear**: turn the drive into a firing rate with a fixed curve. Here it is a **softplus**, `a1 · log(1 + exp(a2 · (drive + a3)))`, roughly zero below a threshold and rising above it.

To fit the curve, the drive is split into 40 bins with equal numbers of frames, and the mean firing rate in each bin is plotted against the mean drive. The softplus is fitted to those points.

In [ ]:
# The cropped spatiotemporal STA: crop pixels × lags.
stac = sta.reshape(WIDTH, HEIGHT, FILTER_LENGTH)[crop].reshape(cropped_sta.size, FILTER_LENGTH)

drive = np.concatenate([ln_drive(stimuli[t], stac, FILTER_LENGTH) for t in range(num_trials)])
counts = spikecounts[FILTER_LENGTH:, :].T.ravel()     # the matching spike counts
centers, binned_rates = binned_response(drive, counts, DT)
ln_parameters = gen_sta_model(
    WIDTH, HEIGHT, spikecounts, crop, *crop_shape, FILTER_LENGTH, num_frames, num_trials,
    stac, STIMULUS_PATH, stimuli=stimuli,
)
print("softplus parameters a1, a2, a3 =", np.round(ln_parameters, 3))

fig, ax = plt.subplots(figsize=(4.5, 3.4))
plot_nonlinearity(centers, binned_rates, lambda d: nonlin(d, *ln_parameters), ax=ax,
                  color=MODEL_COLORS["LN"])
ax.set_title("LN Output Nonlinearity")
fig.tight_layout()
save_figure(fig, FIGURES / "LN Nonlinearity", caption=(
    "Points: mean firing rate in 40 equal-count bins of the LN model's linear drive "
    "(running-noise trials). Line: fitted softplus nonlinearity."))

## 6. Testing the LN model on frozen noise

The real test is the frozen noise: a stimulus the model was not fitted to, with a firing rate measured precisely by averaging many repeats. `get_sta_predictions` runs the frozen stimulus through the LN model and returns the prediction alongside the measured rate.

Two numbers summarise the match:

- **Correlation** (Pearson r): does the prediction go up and down at the right times?
- **Mean squared error**: how far off is it, in Hz²?

No model can match the measured rate perfectly, because even the average of many repeats contains some noise. The **noise ceiling** estimates the best correlation any model could reach: split the repeats into two random halves, correlate the two halves' averages, correct for using only half the data (Spearman–Brown), and take the square root. The **normalized correlation** is the model's correlation divided by that ceiling.

In [ ]:
ln_predictions, _, measured_rate, time = get_sta_predictions(
    CELL, ln_parameters, stac, crop, *crop_shape, DATA_PATH, STIMULUS_PATH,
    filter_length=FILTER_LENGTH, cache_dir=CACHE_DIR,
)
ceiling = correlation_ceiling(spk2[CELL], FILTER_LENGTH)
ln_metrics = prediction_metrics(measured_rate, ln_predictions, FILTER_LENGTH,
                                ceiling=ceiling.max_correlation)
print(f"noise ceiling: r = {ceiling.max_correlation:.3f} (reliability {ceiling.reliability:.3f})")
print(f"LN model: r = {ln_metrics.correlation:.3f}, normalized r = "
      f"{ln_metrics.normalized_correlation:.3f}, MSE = {ln_metrics.mse:.1f} Hz²")

fig, ax = plt.subplots(figsize=(10, 3))
plot_predictions(ln_predictions, measured_rate, time, FILTER_LENGTH, label="LN", ax=ax)
ax.set_title(f"Cell {CELL} Frozen Noise, LN r = {ln_metrics.correlation:.2f}")
fig.tight_layout()
save_figure(fig, FIGURES / "LN Frozen Noise Prediction", caption=(
    f"Measured firing rate of cell {CELL} (mean of {spk2.shape[2]} frozen-noise repeats) "
    f"and the LN model's prediction. The first {FILTER_LENGTH} frames are dropped because "
    f"the filter needs that much history. r = {ln_metrics.correlation:.3f}; noise ceiling "
    f"{ceiling.max_correlation:.3f}."))

## Where next

The LN model treats the receptive field as one linear filter. But ganglion cells pool inputs from bipolar cells, and each of those inputs is rectified on its own before they are summed. Those inputs are the receptive field's **subunits**, and a model that rectifies each one separately can capture responses the LN model misses.

- **Tutorial 2** trains a spike classifier and reads candidate subunits from its hidden units.
- **Tutorial 3** finds subunits with an autoencoder instead.

Both then build a subunit model and compare it with this LN model on the same frozen noise.